In [ ]:
import typing
import bs4
import pickle
from scrappey import Scrappey, ScrappeyTimeoutError
import urllib
import time

SCHOLAR_PROFILE = "qHFA5z4AAAAJ"

In [ ]:
def get(url: str, requestType="browser", timeout=20, timeout_sleep=10) -> str:
    try:
        with Scrappey(
            api_key="vP7OkTdBWXhQoEXO9pCu3sR7EFPIImLYLAuXjncUoJSzibWiAymA2wGdaZTH",
            timeout=timeout,
        ) as scrappey:
            result = scrappey.get(
                url=url,
                requestType=requestType,
                # proxyCountry="UnitedStates",
                automaticallySolveCaptchas=True,
                premiumProxy=True,
                cloudflareBypass=True,
                kasadaBypass=True,
                datadomeBypass=True,
            )

        if result["data"] == "success":
            status = result["solution"]["statusCode"]
            response = result["solution"]["response"]
            print("status =", status, "url =", url)
            if status == 200:
                return response
            else:
                return get(url, requestType="browser")
        elif result["data"] == "error":
            print("error:", result["error"])
            return get(url, requestType="browser")
        else:
            raise Exception(f"unknown error...")

    except ScrappeyTimeoutError:
        print("timeout...")
        time.sleep(timeout_sleep)
        return get(url, requestType="browser")
    except Exception as e:
        raise Exception(f"error: {e}, result={result}")

# Get profile

In [13]:
html = get(
    f"https://scholar.google.com/citations?hl=en&user={SCHOLAR_PROFILE}&pagesize=100"
)
soup = bs4.BeautifulSoup(html)

paper_url_list = [
    "https://scholar.google.com"
    + paper.find(name="a", attrs={"class": "gsc_a_at"}).attrs["href"]
    for paper in soup.find_all(name="tr", attrs={"class": "gsc_a_tr"})
]

print(len(paper_url_list))

for url in paper_url_list:
    print(url)

status = 200 url = https://scholar.google.com/citations?hl=en&user=qHFA5z4AAAAJ&pagesize=100
55
https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:Y0pCki6q_DkC
https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:2osOgNQ5qMEC
https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:bnK-pcrLprsC
https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:Se3iqnhoufwC
https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:IjCSPb-OGe4C
https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:Tyk-4Ss8FVUC
https://scholar.google.com/citations?view_op=view_citation&hl=en

# Get paper cards

In [14]:
def url_to_dict(url, requestType="request"):
    print("url_to_dict", url)
    paper_soup = bs4.BeautifulSoup(get(url, requestType))
    title = paper_soup.find(name="div", attrs={"id": "gsc_oci_title"})
    if title is None:
        print("retry...")
        return url_to_dict(url, requestType="browser")

    paper_dict: typing.Dict[str, bs4.Tag] = {}

    paper_dict["Title"] = title

    key_list = paper_soup.find_all(name="div", attrs={"class": "gsc_oci_field"})
    value_list = paper_soup.find_all(name="div", attrs={"class": "gsc_oci_value"})

    for key, value in zip(key_list, value_list):
        paper_dict[key.text] = value

    return paper_dict

In [15]:
paper_dict_list: typing.List[typing.Dict[str, bs4.Tag]] = []
for url in paper_url_list:
    paper_dict_list.append(url_to_dict(url))

url_to_dict https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:Y0pCki6q_DkC
status = 200 url = https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:Y0pCki6q_DkC
url_to_dict https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:2osOgNQ5qMEC
status = 200 url = https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:2osOgNQ5qMEC
url_to_dict https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:bnK-pcrLprsC
status = 200 url = https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:bnK-pcrLprsC
url_to_dict https://scholar.google.com/citations?view_op=view_citat

In [16]:
pickle.dump(paper_dict_list, open("papers.txt", "wb"))

# Get paper versions

In [17]:
paper_dict_list: typing.List[typing.Dict[str, bs4.Tag]] = pickle.load(
    open("papers.txt", "rb")
)
print(paper_dict_list[0].keys())
for i in range(len(paper_dict_list)):
    print(i, paper_dict_list[i]["Title"].text)

dict_keys(['Title', 'Authors', 'Publication date', 'Journal', 'Volume', 'Issue', 'Pages', 'Publisher', 'Description', 'Total citations', 'Scholar articles'])
0 Stochastic distributed learning with gradient quantization and double-variance reduction
1 Don’t jump through hoops and remove those loops: SVRG and Katyusha are better without the outer loop
2 Acceleration for compressed gradient descent in distributed and federated optimization
3 From local SGD to local fixed-point methods for federated learning
4 RSN: randomized subspace Newton
5 Revisiting stochastic extragradient
6 Optimal and practical algorithms for smooth and strongly convex decentralized optimization
7 Linearly converging error compensated SGD
8 A linearly convergent algorithm for decentralized optimization: Sending less bits for free!
9 Accelerated methods for saddle-point problem
10 Accelerated primal-dual gradient method for smooth and convex-concave saddle-point problems with bilinear coupling
11 Stochastic Newton a

In [18]:
def extract_clusters(scholar_articles: bs4.Tag) -> typing.List[str]:
    merged_snippet_list: bs4.ResultSet = scholar_articles.find_all(
        name="div", attrs={"class": "gsc_oci_merged_snippet"}
    )
    version_url_list: typing.List[str] = []
    for merged_snippet in merged_snippet_list:
        version_href = merged_snippet.find(name="a").attrs["href"]
        version_cluster = urllib.parse.parse_qs(
            urllib.parse.urlparse(version_href).query
        )["cluster"][0]
        version_url = f"https://scholar.google.com/scholar?start=0&num=100&hl=en&cluster={version_cluster}"
        version_url_list.append(version_url)
    return version_url_list


def get_cluster_versions(cluster, requestType="request"):
    cluster_versions = bs4.BeautifulSoup(get(cluster, requestType)).find_all(
        attrs={"class": "gs_rt"}
    )
    if len(cluster_versions) == 0:
        print("retry...")
        return get_cluster_versions(cluster, requestType="browser")
    return cluster_versions


def extract_versions(scholar_articles) -> typing.List[str]:
    cluster_list = extract_clusters(scholar_articles)

    version_list: typing.List[str] = []
    for cluster in cluster_list:
        print("cluster =", cluster)
        cluster_versions = get_cluster_versions(cluster)
        for version in cluster_versions:
            a = version.find(name="a")
            if not a is None:
                version_list.append(a.attrs["href"])

    return version_list

In [43]:
for paper_dict in paper_dict_list:
    print(paper_dict["Title"].text)
    if "Scholar articles" in paper_dict.keys():
        version_list = extract_versions(paper_dict["Scholar articles"])
        print("version count =", len(version_list))
        paper_dict["Version urls"] = version_list
    else:
        paper_dict["Version urls"] = []

Stochastic distributed learning with gradient quantization and double-variance reduction
cluster = https://scholar.google.com/scholar?start=0&num=100&hl=en&cluster=10432066948921138844
error: Service temporarily unavailable, please retry
status = 200 url = https://scholar.google.com/scholar?start=0&num=100&hl=en&cluster=10432066948921138844
version count = 13
Don’t jump through hoops and remove those loops: SVRG and Katyusha are better without the outer loop
cluster = https://scholar.google.com/scholar?start=0&num=100&hl=en&cluster=7006814588298832736
status = 200 url = https://scholar.google.com/scholar?start=0&num=100&hl=en&cluster=7006814588298832736
version count = 8
Acceleration for compressed gradient descent in distributed and federated optimization
cluster = https://scholar.google.com/scholar?start=0&num=100&hl=en&cluster=1371688885190462102
status = 200 url = https://scholar.google.com/scholar?start=0&num=100&hl=en&cluster=1371688885190462102
version count = 16
From local SGD 

In [44]:
pickle.dump(paper_dict_list, open("paper_urls.txt", "wb"))